## Exploratory Data Analysis and Cleanup for Children's Books

In [1]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

/Users/navyajain/Downloads/build/agentic-book-recommendation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 42

In [ ]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

In [ ]:
#print(children_path)

../data/books/goodreads_books_children.json


In [3]:
comic_df = pd.read_json('../data/books/goodreads_books_comics_graphic.json', lines=True)

In [4]:
comic_df.shape

(89411, 29)

In [5]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,1,[],US,,"[{'count': '228', 'name': 'to-read'}, {'count'...",B00NLXQ534,true,4.12,,"[25653153, 25699172, 23530486, 12984185, 25538...",Lillian Ann Cross is forced to live the worst ...,,https://www.goodreads.com/book/show/25742454-t...,"[{'author_id': '8551671', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/25742454-t...,https://s.gr-assets.com/assets/nophoto/book/11...,25742454,1,42749946,The Switchblade Mamma,The Switchblade Mamma
1,2205073346,2,[],US,fre,"[{'count': '2', 'name': 'bd'}, {'count': '2', ...",,false,3.94,,[],"Florence Dupre Latour raconte comment, de son ...",,https://www.goodreads.com/book/show/30128855-c...,"[{'author_id': '3274315', 'role': ''}]",Dargaud,,22,,1,,2016,https://www.goodreads.com/book/show/30128855-c...,https://images.gr-assets.com/books/1462644346m...,30128855,16,50558228,Cruelle,Cruelle
2,,5,"[246830, 362583, 362581, 623032]",US,eng,"[{'count': '493', 'name': 'to-read'}, {'count'...",,false,4.28,,"[13590139, 105963, 207585, 10503130, 4645370, ...",The questions plaguing Captain America's dream...,Hardcover,https://www.goodreads.com/book/show/13571772-c...,"[{'author_id': '37450', 'role': ''}]",Hachette Partworks Ltd.,146,,,,,2012,https://www.goodreads.com/book/show/13571772-c...,https://images.gr-assets.com/books/1333287305m...,13571772,51,102217,Captain America: Winter Soldier (The Ultimate ...,Captain America: Winter Soldier (The Ultimate ...
3,,1,[],US,eng,"[{'count': '222', 'name': 'to-read'}, {'count'...",B06XKGGSB7,true,4.05,B06XKGGSB7,[],The fight for Jason Delgado's life and soul be...,,https://www.goodreads.com/book/show/35452242-b...,"[{'author_id': '16209952', 'role': ''}, {'auth...",,,,,,,,https://www.goodreads.com/book/show/35452242-b...,https://s.gr-assets.com/assets/nophoto/book/11...,35452242,6,54276229,Bounty Hunter 4/3: My Life in Combat from Mari...,Bounty Hunter 4/3: My Life in Combat from Mari...
4,0930289765,6,"[266759, 1096220]",US,en-US,"[{'count': '20', 'name': 'to-read'}, {'count':...",,false,4.06,,[],These are the stories that catapulted Superman...,Hardcover,https://www.goodreads.com/book/show/707611.Sup...,"[{'author_id': '81563', 'role': ''}, {'author_...",DC Comics,272,14,9780930289768,11,,1997,https://www.goodreads.com/book/show/707611.Sup...,https://images.gr-assets.com/books/1307838888m...,707611,51,693886,"Superman Archives, Vol. 2","Superman Archives, Vol. 2"


In [6]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [7]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [8]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [9]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Switchblade Mamma,The Switchblade Mamma,[],US,,,B00NLXQ534,,,42749946,25742454,1,4.12,1,"[{'count': '228', 'name': 'to-read'}, {'count'...",true,,,,,,,,Lillian Ann Cross is forced to live the worst ...,"[{'author_id': '8551671', 'role': ''}]","[25653153, 25699172, 23530486, 12984185, 25538..."
1,Cruelle,Cruelle,[],US,fre,2205073346,,,,50558228,30128855,2,3.94,16,"[{'count': '2', 'name': 'bd'}, {'count': '2', ...",false,,,Dargaud,22,1,2016,,"Florence Dupre Latour raconte comment, de son ...","[{'author_id': '3274315', 'role': ''}]",[]
2,Captain America: Winter Soldier (The Ultimate ...,Captain America: Winter Soldier (The Ultimate ...,"[246830, 362583, 362581, 623032]",US,eng,,,,,102217,13571772,5,4.28,51,"[{'count': '493', 'name': 'to-read'}, {'count'...",false,Hardcover,146,Hachette Partworks Ltd.,,,2012,,The questions plaguing Captain America's dream...,"[{'author_id': '37450', 'role': ''}]","[13590139, 105963, 207585, 10503130, 4645370, ..."
3,Bounty Hunter 4/3: My Life in Combat from Mari...,Bounty Hunter 4/3: My Life in Combat from Mari...,[],US,eng,,B06XKGGSB7,B06XKGGSB7,,54276229,35452242,1,4.05,6,"[{'count': '222', 'name': 'to-read'}, {'count'...",true,,,,,,,,The fight for Jason Delgado's life and soul be...,"[{'author_id': '16209952', 'role': ''}, {'auth...",[]
4,"Superman Archives, Vol. 2","Superman Archives, Vol. 2","[266759, 1096220]",US,en-US,0930289765,,,9780930289768,693886,707611,6,4.06,51,"[{'count': '20', 'name': 'to-read'}, {'count':...",false,Hardcover,272,DC Comics,14,11,1997,,These are the stories that catapulted Superman...,"[{'author_id': '81563', 'role': ''}, {'author_...",[]


In [10]:
df['clean_title'] = df['title_without_series']

### Detecting languages

In [11]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [12]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 29614/29614 [01:02<00:00, 470.89it/s]


In [13]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,The Switchblade Mamma,The Switchblade Mamma,[],US,,,B00NLXQ534,,,42749946,25742454,1,4.12,1,"[{'count': '228', 'name': 'to-read'}, {'count'...",true,,,,,,,,Lillian Ann Cross is forced to live the worst ...,"[{'author_id': '8551671', 'role': ''}]","[25653153, 25699172, 23530486, 12984185, 25538...",The Switchblade Mamma,eng,The Switchblade Mamma Lillian Ann Cross is for...,eng
1,Cruelle,Cruelle,[],US,fre,2205073346,,,,50558228,30128855,2,3.94,16,"[{'count': '2', 'name': 'bd'}, {'count': '2', ...",false,,,Dargaud,22,1,2016,,"Florence Dupre Latour raconte comment, de son ...","[{'author_id': '3274315', 'role': ''}]",[],Cruelle,fre,"Cruelle Florence Dupre Latour raconte comment,...",nan
2,Captain America: Winter Soldier (The Ultimate ...,Captain America: Winter Soldier (The Ultimate ...,"[246830, 362583, 362581, 623032]",US,eng,,,,,102217,13571772,5,4.28,51,"[{'count': '493', 'name': 'to-read'}, {'count'...",false,Hardcover,146,Hachette Partworks Ltd.,,,2012,,The questions plaguing Captain America's dream...,"[{'author_id': '37450', 'role': ''}]","[13590139, 105963, 207585, 10503130, 4645370, ...",Captain America: Winter Soldier (The Ultimate ...,eng,Captain America: Winter Soldier (The Ultimate ...,nan
3,Bounty Hunter 4/3: My Life in Combat from Mari...,Bounty Hunter 4/3: My Life in Combat from Mari...,[],US,eng,,B06XKGGSB7,B06XKGGSB7,,54276229,35452242,1,4.05,6,"[{'count': '222', 'name': 'to-read'}, {'count'...",true,,,,,,,,The fight for Jason Delgado's life and soul be...,"[{'author_id': '16209952', 'role': ''}, {'auth...",[],Bounty Hunter 4/3: My Life in Combat from Mari...,eng,Bounty Hunter 4/3: My Life in Combat from Mari...,nan
4,"Superman Archives, Vol. 2","Superman Archives, Vol. 2","[266759, 1096220]",US,en-US,0930289765,,,9780930289768,693886,707611,6,4.06,51,"[{'count': '20', 'name': 'to-read'}, {'count':...",false,Hardcover,272,DC Comics,14,11,1997,,These are the stories that catapulted Superman...,"[{'author_id': '81563', 'role': ''}, {'author_...",[],"Superman Archives, Vol. 2",eng,"Superman Archives, Vol. 2 These are the storie...",nan


In [14]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [15]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [16]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Switchblade Mamma,[],eng,,,42749946,25742454,1,4.12,1,"[{'count': '228', 'name': 'to-read'}, {'count'...",,,,,,,,Lillian Ann Cross is forced to live the worst ...,"[{'author_id': '8551671', 'role': ''}]","[25653153, 25699172, 23530486, 12984185, 25538..."
1,Cruelle,[],fre,2205073346,,50558228,30128855,2,3.94,16,"[{'count': '2', 'name': 'bd'}, {'count': '2', ...",,,Dargaud,22,1,2016,,"Florence Dupre Latour raconte comment, de son ...","[{'author_id': '3274315', 'role': ''}]",[]
2,Captain America: Winter Soldier (The Ultimate ...,"[246830, 362583, 362581, 623032]",eng,,,102217,13571772,5,4.28,51,"[{'count': '493', 'name': 'to-read'}, {'count'...",Hardcover,146,Hachette Partworks Ltd.,,,2012,,The questions plaguing Captain America's dream...,"[{'author_id': '37450', 'role': ''}]","[13590139, 105963, 207585, 10503130, 4645370, ..."
3,Bounty Hunter 4/3: My Life in Combat from Mari...,[],eng,,,54276229,35452242,1,4.05,6,"[{'count': '222', 'name': 'to-read'}, {'count'...",,,,,,,,The fight for Jason Delgado's life and soul be...,"[{'author_id': '16209952', 'role': ''}, {'auth...",[]
4,"Superman Archives, Vol. 2","[266759, 1096220]",eng,0930289765,9780930289768,693886,707611,6,4.06,51,"[{'count': '20', 'name': 'to-read'}, {'count':...",Hardcover,272,DC Comics,14,11,1997,,These are the stories that catapulted Superman...,"[{'author_id': '81563', 'role': ''}, {'author_...",[]


In [17]:
df.shape

(89411, 21)

In [18]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [19]:
df.shape

(57537, 21)

In [20]:
df['format'].value_counts().shape

(171,)

In [21]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['', 'Hardcover', 'Paperback Manga', 'Paperback', 'Kindle Edition', 'ebook', 'hardcover', 'Paperback comic book', 'Comic Book', 'Mass Market Paperback', 'Comic', 'Digital Comic', 'Unknown Binding', 'Nook', 'Library Binding', 'Webtoon', 'Comics', 'Book', 'Audio', 'Audible Audio', 'Hardcover with dust jacket', 'Custom Handmade Binding', 'Spiral-bound', 'Audio CD', 'free manga scanlation', 'Trade Paperback', 'Graphic Novels', 'FC', 'Slipcased Hardcover', 'Online', 'webtoon', 'Board book', 'Unbound', 'Single Issue', 'comic', 'paperback', 'One-Shot Comic', 'Marvel Comics', 'Comixology Edition', 'Digital', 'Comic book', 'comics', 'The Walking Dead - Single Issues #146', 'Over-sized Soft-cover', 'Audiobook', 'Da Xing Ben', 'Paperback - Manga', 'Issue', 'Softcover', 'Flexiback', 'The Walking Dead - Single Issues #135', 'Pamphlet', 'Graphic Novel', 'Webcomic/Manhua', 'English scanlation', 'Print Comic', 'Online Comic', 'PDF', 'Magazine', 'CD-ROM', 'staple-bound', 'Board Book', 'Hard cover', 'Ha

In [22]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard physical books
    "Hardcover",
    "hardcover",
    "Hardcover with dust jacket",
    "Hard cover",
    "Hard Cover",
    "Hardback",
    "Paperback",
    "paperback",
    "Mass Market Paperback",
    "Trade Paperback",
    "trade paperback",
    "Trade paperback",
    "Trade Paper",
    "Over-sized Soft-cover",
    "Oversized PB",
    "Softcover",
    "soft cover",
    "Soft Cover",
    "mini-paperback",
    "Broche",
    "Flexiback",
    "Deluxe Flexicover",
    "Deluxe Softcover",
    "Trade Cloth",
    "Leather Bound",
    "Library Binding",
    "Custom Handmade Binding",
    "Spiral-bound",
    "Spiral",
    "Unbound",
    "Pamphlet",
    "staple-bound",
    "Saddle-stitched",
    "Saddle stitched",
    "saddle stitch",
    "Squarebound",

    # Board / novelty / card books
    "Board book",
    "Board Book",
    "Card Book",
    "Novelty Book",

    # Manga
    "Manga",
    "manga",
    "Paperback Manga",
    "Paperback - Manga",
    "Manga Online",

    # Comics / comic books
    "Comic",
    "comic",
    "Comics",
    "comics",
    "Comic Book",
    "Comic book",
    "comic book",
    "comic books",
    "Comicbook",
    "Paperback comic book",
    "Print Comic",
    "Comic Book Issue",
    "Digital Comic Book",
    "One-Shot Comic",
    "Prestige One-Shot",
    "Serial Comic",
    "Single Issue",
    "issue",
    "Issue",
    "Single issue comic",
    "Single Issue Comicbook",
    "Graphic novel issue",
    "Horror Comic",
    "Marvel Comics",
    "deluxe comic",
    "FC",

    # Graphic novels
    "Graphic Novel",
    "Graphic novel",
    "graphic novel",
    "GRAPHIC NOVEL",
    "Grapic Novel",
    "Graphic Novels",
    "Graphic",
    "Graphic Anthology",
    "Graphic Novel Paperback",
    "Graphic Paperback",
    "Trade Paperback Graphic Novel",
    "Trade Paperback - hardback",
    "Graphic Novel Trade Hardcover",
    "Paperback graphic novel",
    "Paperback Graphic Novel",
    "Paperpack Graphic Novels",
    "Comic Trade Paperback",
    "TPB",
    "Hardcover Comic",
    "Kindle Graphic Novel",

    # Webcomic / webtoon / manhua / manhwa
    "Webtoon",
    "webtoon",
    "Webcomic",
    "Webcomics",
    "Web Comic",
    "Web Comic/ Manhua",
    "Webcomic/Manhua",
    "Online Comic",
    "online comic",
    "Online Comic - Webcomic",
    "Online eBook",
    "Online Manga Series",
    "online graphic novel - complete",
    "web",
    "manhwa",

    # Digital readable formats
    "Kindle Edition",
    "Kindle Edition, with panel zoom",
    "ebook",
    "Nook",
    "Online",
    "Digital",
    "digital",
    "Digital Comic",
    "Digital comic",
    "Comixology Edition",
    "Comixology",
    "Comixology Guided View",
    "PDF",
    "PDF DIGITAL EDITION",

    # Collections / special readable editions
    "Book",
    "Box Set",
    "Boxed Set",
    "Boxed-Set",
    "collection",
    "Slipcased Hardcover",
    "Slipcase Hardcover",
    "Slipcased hardcover (1-3)",
    "Oversized Slipcased Hardcover",
    "Prestige",
    "Album",
    "artist's book",
    "zine",
    "Zine",
    "Magazine",
    "Tabloid",
    "Da Xing Ben",
    "A4 Softcover",
    "13&quot; by 10&quot; softcover",
}

# 2. Define formats to drop
drop_formats = {
    # Missing / unknown
    "",
    "Unknown Binding",

    # Audio
    "Audio",
    "Audible Audio",
    "Audiobook",
    "Audio CD",
    "Audio Cassette",
    "GraphicAudio",

    # Video / software / discs
    "CD-ROM",
    "DVD-ROM",

    # Pirated / scanlation-ish / not clean catalog source
    "free manga scanlation",
    "English scanlation",

    # Non-readable / merch / weird object
    "Vinyl",
    "Hand-bound Art Portfolio w/7&quot; Vinyl Record",
    "Cardstock",
    "Motion Book",

    # Specific issue labels that are not useful as formats
    "The Walking Dead - Single Issues #146",
    "The Walking Dead - Single Issues #135",
    "The Walking Dead - Single Issues #144",
    "The Walking Dead - Single Issues #143",
    "The Walking Dead - Single Issues #137",
    "The Walking Dead - Single Issues #134",
    "The Walking Dead - Single Issues #141",
    "The Walking Dead - Single Issues #136",
    "The Walking Dead - Single Issues #140",
    "The Walking Dead - Single Issues #139",
    "The Walking Dead - Single Issues #142",
    "The Walking Dead - Single Issues #138",
    "(The Walking Dead - Single Issues #132)",

    # Not really a format
    "17,2 x 19,2 x 12,2 cm",
}

# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      42604
other     14929
review        4
Name: count, dtype: int64

In [23]:
df = df[df["format_group"] == "text"].copy()

In [24]:
df["format_group"].value_counts()

format_group
text    42604
Name: count, dtype: int64

In [25]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
2,Captain America: Winter Soldier (The Ultimate ...,"[246830, 362583, 362581, 623032]",eng,,,102217,13571772,5,4.28,51,"[{'count': '493', 'name': 'to-read'}, {'count'...",Hardcover,146,Hachette Partworks Ltd.,,,2012,,The questions plaguing Captain America's dream...,"[{'author_id': '37450', 'role': ''}]","[13590139, 105963, 207585, 10503130, 4645370, ...",Hardcover,text
4,"Superman Archives, Vol. 2","[266759, 1096220]",eng,0930289765,9780930289768,693886,707611,6,4.06,51,"[{'count': '20', 'name': 'to-read'}, {'count':...",Hardcover,272,DC Comics,14,11,1997,,These are the stories that catapulted Superman...,"[{'author_id': '81563', 'role': ''}, {'author_...",[],Hardcover,text
5,"A.I. Revolution, Vol. 1",[910384],eng,1933617640,9781933617640,2256459,2250580,7,3.44,46,"[{'count': '15', 'name': 'to-read'}, {'count':...",Paperback Manga,206,Go! Comi,2,1,2007,,"Like everyone else in the future, Sui's used t...","[{'author_id': '1015982', 'role': ''}]",[],Paperback Manga,text
6,"War Stories, Volume 3",[834955],eng,1592912729,9781592912728,47077783,27036536,9,4.15,39,"[{'count': '47', 'name': 'to-read'}, {'count':...",Paperback,224,Avatar Press,26,1,2016,,PRODUCT DESCRIPTION: The first new volume of G...,"[{'author_id': '14965', 'role': ''}, {'author_...",[],Paperback,text
7,"Crossed, Volume 15",[961379],eng,1592912737,9781592912735,47077784,27036537,2,3.16,38,"[{'count': '24', 'name': 'to-read'}, {'count':...",Paperback,160,Avatar Press,8,3,2016,,Comics horror veteran Mike Wolfer writes and i...,"[{'author_id': '24594', 'role': ''}]",[],Paperback,text


In [26]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [27]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [28]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
2,Captain America: Winter Soldier (The Ultimate ...,"[246830, 362583, 362581, 623032]",eng,,,102217,13571772,5,4.28,51,"[{'count': '493', 'name': 'to-read'}, {'count'...",Hardcover,146,Hachette Partworks Ltd.,2012,The questions plaguing Captain America's dream...,"[{'author_id': '37450', 'role': ''}]","[13590139, 105963, 207585, 10503130, 4645370, ..."
4,"Superman Archives, Vol. 2","[266759, 1096220]",eng,0930289765,9780930289768,693886,707611,6,4.06,51,"[{'count': '20', 'name': 'to-read'}, {'count':...",Hardcover,272,DC Comics,1997,These are the stories that catapulted Superman...,"[{'author_id': '81563', 'role': ''}, {'author_...",[]
5,"A.I. Revolution, Vol. 1",[910384],eng,1933617640,9781933617640,2256459,2250580,7,3.44,46,"[{'count': '15', 'name': 'to-read'}, {'count':...",Paperback Manga,206,Go! Comi,2007,"Like everyone else in the future, Sui's used t...","[{'author_id': '1015982', 'role': ''}]",[]
6,"War Stories, Volume 3",[834955],eng,1592912729,9781592912728,47077783,27036536,9,4.15,39,"[{'count': '47', 'name': 'to-read'}, {'count':...",Paperback,224,Avatar Press,2016,PRODUCT DESCRIPTION: The first new volume of G...,"[{'author_id': '14965', 'role': ''}, {'author_...",[]
7,"Crossed, Volume 15",[961379],eng,1592912737,9781592912735,47077784,27036537,2,3.16,38,"[{'count': '24', 'name': 'to-read'}, {'count':...",Paperback,160,Avatar Press,2016,Comics horror veteran Mike Wolfer writes and i...,"[{'author_id': '24594', 'role': ''}]",[]


In [29]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [30]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [31]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [32]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [33]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,797,The League Of Extraordinary Gentleman Vol. 1,"[{'author_id': '3961', 'role': ''}, {'author_i...","[[185613, 458201]]",eng,"[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...","[2002.0, 2001.0, 2009.0]",".As the Victorian era draws to a close, Allan ...","[{'count': '2020', 'name': 'graphic-novels'}, ...","[[22421, 6660561, 900750, 300678, 60112, 79426...",1152,40900,3.93,192.0
1,1027,"Kabuki, Vol. 1: Circle of Blood","[{'author_id': '10455', 'role': ''}]",[[248363]],eng,"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",[2010.0],Finally back in hardcover after ten years! The...,"[{'count': '1438', 'name': 'to-read'}, {'count...","[[87232, 100581, 521098, 864065, 1074937, 1264...",108,1665,4.19,272.0
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,"[{'author_id': '5112', 'role': ''}]",[],eng,"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]","[2001.0, 2000.0, 2003.0]",This first book from Chicago author Chris Ware...,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[[43575, 43555, 405630, 76574, 144143, 33472, ...",879,17341,4.10,380.0
3,2647,"The Kindly Ones (The Sandman, #9)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144460]],eng,"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...","[1996.0, 1999.0, 2012.0]",THE SANDMAN is the most acclaimed and award-wi...,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[[151872, 21324, 109243, 22422, 102956]]",649,32994,4.59,368.0
4,2648,"The Doll's House (The Sandman, #2)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144461]],eng,"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...","[1995.0, 2013.0, 2011.0, 2010.0, 1999.0, 1990....",NEW YORK TIMES bestselling author Neil Gaiman'...,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[[437094, 21330, 109244, 22422, 184040]]",1207,55387,4.44,232.0


In [34]:
df = books_work_df.copy()

In [35]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,797,The League Of Extraordinary Gentleman Vol. 1,"[{'author_id': '3961', 'role': ''}, {'author_i...","[[185613, 458201]]",eng,"[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...","[2002.0, 2001.0, 2009.0]",".As the Victorian era draws to a close, Allan ...","[{'count': '2020', 'name': 'graphic-novels'}, ...","[[22421, 6660561, 900750, 300678, 60112, 79426...",1152,40900,3.93,192.0
1,1027,"Kabuki, Vol. 1: Circle of Blood","[{'author_id': '10455', 'role': ''}]",[[248363]],eng,"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",[2010.0],Finally back in hardcover after ten years! The...,"[{'count': '1438', 'name': 'to-read'}, {'count...","[[87232, 100581, 521098, 864065, 1074937, 1264...",108,1665,4.19,272.0
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,"[{'author_id': '5112', 'role': ''}]",[],eng,"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]","[2001.0, 2000.0, 2003.0]",This first book from Chicago author Chris Ware...,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[[43575, 43555, 405630, 76574, 144143, 33472, ...",879,17341,4.10,380.0
3,2647,"The Kindly Ones (The Sandman, #9)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144460]],eng,"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...","[1996.0, 1999.0, 2012.0]",THE SANDMAN is the most acclaimed and award-wi...,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[[151872, 21324, 109243, 22422, 102956]]",649,32994,4.59,368.0
4,2648,"The Doll's House (The Sandman, #2)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144461]],eng,"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...","[1995.0, 2013.0, 2011.0, 2010.0, 1999.0, 1990....",NEW YORK TIMES bestselling author Neil Gaiman'...,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[[437094, 21330, 109244, 22422, 184040]]",1207,55387,4.44,232.0


In [36]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description              2348
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                2395
dtype: int64

In [37]:
df.shape

(36593, 17)

In [38]:
df = df[df["description"].notna()].copy()

In [39]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description                 0
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                1903
dtype: int64

In [40]:
df.shape

(34245, 17)

In [41]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,797,The League Of Extraordinary Gentleman Vol. 1,"[{'author_id': '3961', 'role': ''}, {'author_i...","[[185613, 458201]]",eng,"[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...","[2002.0, 2001.0, 2009.0]",".As the Victorian era draws to a close, Allan ...","[{'count': '2020', 'name': 'graphic-novels'}, ...","[[22421, 6660561, 900750, 300678, 60112, 79426...",1152,40900,3.93,192.0
1,1027,"Kabuki, Vol. 1: Circle of Blood","[{'author_id': '10455', 'role': ''}]",[[248363]],eng,"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",[2010.0],Finally back in hardcover after ten years! The...,"[{'count': '1438', 'name': 'to-read'}, {'count...","[[87232, 100581, 521098, 864065, 1074937, 1264...",108,1665,4.19,272.0
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,"[{'author_id': '5112', 'role': ''}]",[],eng,"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]","[2001.0, 2000.0, 2003.0]",This first book from Chicago author Chris Ware...,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[[43575, 43555, 405630, 76574, 144143, 33472, ...",879,17341,4.10,380.0
3,2647,"The Kindly Ones (The Sandman, #9)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144460]],eng,"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...","[1996.0, 1999.0, 2012.0]",THE SANDMAN is the most acclaimed and award-wi...,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[[151872, 21324, 109243, 22422, 102956]]",649,32994,4.59,368.0
4,2648,"The Doll's House (The Sandman, #2)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144461]],eng,"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...","[1995.0, 2013.0, 2011.0, 2010.0, 1999.0, 1990....",NEW YORK TIMES bestselling author Neil Gaiman'...,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[[437094, 21330, 109244, 22422, 184040]]",1207,55387,4.44,232.0


In [42]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [43]:
df.shape

(34245, 15)

In [44]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,797,The League Of Extraordinary Gentleman Vol. 1,"[{'author_id': '3961', 'role': ''}, {'author_i...","[[185613, 458201]]","[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...",".As the Victorian era draws to a close, Allan ...","[{'count': '2020', 'name': 'graphic-novels'}, ...","[[22421, 6660561, 900750, 300678, 60112, 79426...",1152,40900,3.93,192.0
1,1027,"Kabuki, Vol. 1: Circle of Blood","[{'author_id': '10455', 'role': ''}]",[[248363]],"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",Finally back in hardcover after ten years! The...,"[{'count': '1438', 'name': 'to-read'}, {'count...","[[87232, 100581, 521098, 864065, 1074937, 1264...",108,1665,4.19,272.0
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,"[{'author_id': '5112', 'role': ''}]",[],"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]",This first book from Chicago author Chris Ware...,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[[43575, 43555, 405630, 76574, 144143, 33472, ...",879,17341,4.10,380.0
3,2647,"The Kindly Ones (The Sandman, #9)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144460]],"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...",THE SANDMAN is the most acclaimed and award-wi...,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[[151872, 21324, 109243, 22422, 102956]]",649,32994,4.59,368.0
4,2648,"The Doll's House (The Sandman, #2)","[{'author_id': '1221698', 'role': ''}, {'autho...",[[144461]],"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...",NEW YORK TIMES bestselling author Neil Gaiman'...,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[[437094, 21330, 109244, 22422, 184040]]",1207,55387,4.44,232.0


In [45]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [46]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [47]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [48]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [49]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [50]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [51]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '2020', 'name': 'graphic-novels'}, {'count': '711', 'name': 'to-read'}, {'count': '424', 'name': 'fantasy'}, {'count': '345', 'name': 'cómics'}, {'count': '319', 'name': 'fiction'}, {'count': '266', 'name': 'currently-reading'}, {'count': '197', 'name': 'favorites'}, {'count': '191', 'name': 'comics-graphic-novels'}, {'count': '176', 'name': 'comic-books'}, {'count': '153', 'name': 'science-fiction'}, {'count': '133', 'name': 'historical-fiction'}, {'count': '130', 'name': 'graphic-novel'}, {'count': '125', 'name': 'graphic-novels-comics'}, {'count': '125', 'name': 'comic'}, {'count': '119', 'name': 'owned'}, {'count': '108', 'name': 'sci-fi'}, {'count': '96', 'name': 'graphic'}, {'count': '82', 'name': 'comics-and-graphic-novels'}, {'count': '71', 'name': 'library'}, {'count': '66', 'name': 'series'}, {'count': '61', 'name': 'alan-moore'}, {'count': '54', 'name': 'alternate-history'}, {'count': '49', 'name': 'graphic-novels-and-comics'}, {'count': '48', 'name': 'boo

In [52]:
int_df = df[['popular_shelves']]

In [53]:
int_df.head()

,popular_shelves
0,"[{'count': '2020', 'name': 'graphic-novels'}, ..."
1,"[{'count': '1438', 'name': 'to-read'}, {'count..."
2,"[{'count': '13031', 'name': 'to-read'}, {'coun..."
3,"[{'count': '12039', 'name': 'to-read'}, {'coun..."
4,"[{'count': '17031', 'name': 'to-read'}, {'coun..."


In [54]:
int_df.shape

(34245, 1)

In [55]:
int_df.to_csv('../data/intermediate/comic_shelves.csv', index=False)

In [56]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/comic_tag_mapping.csv')

In [57]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,11576449,33824
1,manga,manga,keep,1220908,9486
2,graphic-novels,graphic-novel,keep,993772,29246
3,comics,comics,keep,930581,29498
4,graphic-novel,graphic-novel,keep,429420,24545
5,currently-reading,status,drop,290338,22343
6,fantasy,fantasy,keep,239642,14544
7,favorites,review,drop,201407,17364
8,fiction,fiction,keep,168872,20329
9,owned,ownership,drop,110850,19972


In [58]:
vocab_df = shelves.copy()

In [59]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [60]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,11576449,33824
1,manga,manga,keep,1220908,9486
2,graphic-novels,graphic-novel,keep,993772,29246
3,comics,comics,keep,930581,29498
4,graphic-novel,graphic-novel,keep,429420,24545


In [61]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [62]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [63]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '2020', 'name': 'graphic-novels'}, ...","[{'count': '2020', 'name': 'graphic-novel'}, {..."
1,"[{'count': '1438', 'name': 'to-read'}, {'count...","[{'count': '170', 'name': 'graphic-novel'}, {'..."
2,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[{'count': '977', 'name': 'graphic-novel'}, {'..."
3,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[{'count': '1940', 'name': 'graphic-novel'}, {..."
4,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[{'count': '3269', 'name': 'graphic-novel'}, {..."


In [64]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '2020', 'name': 'graphic-novels'}, {'count': '711', 'name': 'to-read'}, {'count': '424', 'name': 'fantasy'}, {'count': '345', 'name': 'cómics'}, {'count': '319', 'name': 'fiction'}, {'count': '266', 'name': 'currently-reading'}, {'count': '197', 'name': 'favorites'}, {'count': '191', 'name': 'comics-graphic-novels'}, {'count': '176', 'name': 'comic-books'}, {'count': '153', 'name': 'science-fiction'}, {'count': '133', 'name': 'historical-fiction'}, {'count': '130', 'name': 'graphic-novel'}, {'count': '125', 'name': 'graphic-novels-comics'}, {'count': '125', 'name': 'comic'}, {'count': '119', 'name': 'owned'}, {'count': '108', 'name': 'sci-fi'}, {'count': '96', 'name': 'graphic'}, {'count': '82', 'name': 'comics-and-graphic-novels'}, {'count': '71', 'name': 'library'}, {'count': '66', 'name': 'series'}, {'count': '61', 'name': 'alan-moore'}, {'count': '54', 'name': 'alternate-history'}, {'count': '49', 'name': 'graphic-novels-and-comics'}, {'count': 

In [65]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [66]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [67]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '282', 'name': 'graphic-novel'}, {'count': '175', 'name': 'comics'}, {'count': '19', 'name': 'fiction'}, {'count': '16', 'name': 'fantasy'}, {'count': '12', 'name': 'manga'}, {'count': '12', 'name': 'science-fiction'}, {'count': '5', 'name': 'adult'}, {'count': '5', 'name': 'image'}, {'count': '4', 'name': 'art'}, {'count': '3', 'name': 'crime'}, {'count': '4', 'name': 'adventure'}, {'count': '2', 'name': 'mystery'}, {'count': '2', 'name': 'dystopia'}, {'count': '4', 'name': 'superhero'}, {'count': '2', 'name': 'young-adult'}, {'count': '1', 'name': 'multicultural'}]


In [68]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '371', 'name': 'comics'}
{'count': '331', 'name': 'graphic-novel'}
{'count': '184', 'name': 'dc'}
{'count': '54', 'name': 'superhero'}
{'count': '23', 'name': 'fiction'}
{'count': '30', 'name': 'science-fiction'}
{'count': '14', 'name': 'dystopia'}
{'count': '4', 'name': 'fantasy'}
{'count': '6', 'name': 'adventure'}
{'count': '3', 'name': 'steampunk'}
{'count': '4', 'name': 'manga'}
{'count': '2', 'name': 'thriller'}
{'count': '2', 'name': 'crime'}
{'count': '2', 'name': 'school'}
{'count': '2', 'name': 'mystery'}
{'count': '2', 'name': 'adult'}


In [69]:
df.shape

(34245, 18)

In [70]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,797,The League Of Extraordinary Gentleman Vol. 1,"[{'author_id': '3961', 'role': ''}, {'author_i...","[185613, 458201]","[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...",".As the Victorian era draws to a close, Allan ...","[{'count': '2020', 'name': 'graphic-novels'}, ...","[22421, 6660561, 900750, 300678, 60112, 79426,...",1152,40900,3.93,192.0,"[3961, 61919]",0,"[{'count': '2893', 'name': 'graphic-novel'}, {..."
1,1027,"Kabuki, Vol. 1: Circle of Blood","[{'author_id': '10455', 'role': ''}]",[248363],"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",Finally back in hardcover after ten years! The...,"[{'count': '1438', 'name': 'to-read'}, {'count...","[87232, 100581, 521098, 864065, 1074937, 12645...",108,1665,4.19,272.0,[10455],0,"[{'count': '282', 'name': 'graphic-novel'}, {'..."
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,"[{'author_id': '5112', 'role': ''}]",[],"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]",This first book from Chicago author Chris Ware...,"[{'count': '13031', 'name': 'to-read'}, {'coun...","[43575, 43555, 405630, 76574, 144143, 33472, 4...",879,17341,4.10,380.0,[5112],0,"[{'count': '1778', 'name': 'graphic-novel'}, {..."
3,2647,The Kindly Ones,"[{'author_id': '1221698', 'role': ''}, {'autho...",[144460],"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...",THE SANDMAN is the most acclaimed and award-wi...,"[{'count': '12039', 'name': 'to-read'}, {'coun...","[151872, 21324, 109243, 22422, 102956]",649,32994,4.59,368.0,"[1221698, 40292, 89468, 19291, 188344, 2896686...",0,"[{'count': '3322', 'name': 'graphic-novel'}, {..."
4,2648,The Doll's House,"[{'author_id': '1221698', 'role': ''}, {'autho...",[144461],"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...",NEW YORK TIMES bestselling author Neil Gaiman'...,"[{'count': '17031', 'name': 'to-read'}, {'coun...","[437094, 21330, 109244, 22422, 184040]",1207,55387,4.44,232.0,"[1221698, 7271, 52930, 12726, 10224, 554069, 1...",0,"[{'count': '5758', 'name': 'graphic-novel'}, {..."


In [71]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [72]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [73]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,797,The League Of Extraordinary Gentleman Vol. 1,"[3961, 61919]","[185613, 458201]","[1840232218, 1563896656, 1840233028, 1563898586]","[9781840232219, 9781563896651, 9781840233025, ...","[3352661, 620786, 107013, 297627]","[Titan Books, America's Best Comics, America's...",".As the Victorian era draws to a close, Allan ...","[22421, 6660561, 900750, 300678, 60112, 79426,...",1152,40900,3.93,192.0,0,"[{'count': '2893', 'name': 'graphic-novel'}, {..."
1,1027,"Kabuki, Vol. 1: Circle of Blood",[10455],[248363],"[1887279806, 0785150021]","[9781887279802, 9780785150022]","[89816, 9172685]","[Image Comics, Marvel]",Finally back in hardcover after ten years! The...,"[87232, 100581, 521098, 864065, 1074937, 12645...",108,1665,4.19,272.0,0,"[{'count': '282', 'name': 'graphic-novel'}, {'..."
2,1315,Jimmy Corrigan: The Smartest Kid on Earth,[5112],[],"[0224062107, 0375404538, 0375714545, 0224063979]","[9780224062107, 9780375404535, 9780375714542, ...","[863050, 412264, 321821, 34072]","[Jonathan Cape, Pantheon]",This first book from Chicago author Chris Ware...,"[43575, 43555, 405630, 76574, 144143, 33472, 4...",879,17341,4.10,380.0,0,"[{'count': '1778', 'name': 'graphic-novel'}, {..."
3,2647,The Kindly Ones,"[1221698, 40292, 89468, 19291, 188344, 2896686...",[144460],"[1852866837, 1563892057, 1417686189, 156389204...","[9781852866839, 9781563892059, 9781417686186, ...","[825059, 437382, 13405305, 1023049, 71252, 132...","[Titan Books Ltd, Vertigo, Turtleback Books, D...",THE SANDMAN is the most acclaimed and award-wi...,"[151872, 21324, 109243, 22422, 102956]",649,32994,4.59,368.0,0,"[{'count': '3322', 'name': 'graphic-novel'}, {..."
4,2648,The Doll's House,"[1221698, 7271, 52930, 12726, 10224, 554069, 1...",[144461],"[1852862920, 0930289595, 1848568193, 156389225...","[9781852862923, 9780930289591, 9781848568198, ...","[825062, 92062, 27826345, 17213611, 9334389, 2...","[Titan Books, Ltd., Vertigo, Vertigo; Reprint ...",NEW YORK TIMES bestselling author Neil Gaiman'...,"[437094, 21330, 109244, 22422, 184040]",1207,55387,4.44,232.0,0,"[{'count': '5758', 'name': 'graphic-novel'}, {..."


In [74]:
df.shape

(34245, 16)

In [75]:
df.to_csv('../data/intermediate/books-cleaned/comics.csv', index=False)